# NB07 — Q8 Controlled Ablation: Build02 vs Build03-A vs Build03-B
**Scientific AI Builder · Regulatory Agent**

## Hypothesis
- **Build02**: reuse frozen NB04b Static RAG answer as baseline (no fresh API call).
- **Build03-A**: same fixed Agent/model/context and frozen NB03-equivalent retriever, **search_regulation only**.
- **Build03-B**: same fixed Agent/model/context and retriever, **search_regulation + check_current_law**.

**Two comparison levels:**
1. **Cost-saving exploratory**: compare new A with *existing* NB06 v3 B; this **does not isolate the tool effect rigorously**, because B used an earlier tool-specific instruction prompt and was executed at a different time.
2. **Matched A/B run**: run A and B with **identical instructions** and model config; only tool availability changes. This is the preferred controlled comparison, but new paid calls and live Cellar requests occur. Keep both results and log invocation order; one run is pilot evidence, not statistically robust proof.

Always run notebook from the repository root. Do not overwrite NB06 official results.

## 0–7. Reuse the **exact frozen NB06 v3 infrastructure**
The following cells are directly copied from the supplied NB06 v3 to preserve baseline loading, corpus, embedding model, BM25 tokenization, RRF ranking, parity gate, 5-law registry, Cellar SPARQL client and offline tests. These are not a new retriever implementation.

In [ ]:
# Only when packages are missing:
# %pip install openai-agents sentence-transformers rank-bm25 numpy pandas requests beautifulsoup4

In [1]:
import os, re, json, hashlib, platform
from datetime import datetime, timezone, date
from pathlib import Path
from getpass import getpass
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from agents import Agent, Runner, ModelSettings, function_tool

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Run from Regulatory_agent repository root (data/ missing)."
MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"
print("Python:", platform.python_version(), "| Model:", MODEL_NAME)
print("API key available:", bool(os.getenv("OPENAI_API_KEY")))
print("API key is not required until the Q8 official run.")

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Python: 3.13.5 | Model: gpt-6-luna
API key available: False
API key is not required until the Q8 official run.


## 1. Load and fingerprint the **frozen** Build02 / NB04b assets

All compared outputs are loaded rather than regenerated. The Q8 checklist is **evaluation-only**, never given to the agent. Separate run artifacts use `build03_nb06_` prefixes.

In [2]:
FILES = {
    "chunks": DATA_DIR / "chunks_legal_aware.json",
    "benchmark": DATA_DIR / "benchmark_questions_v1.json",
    "retrieval_config": DATA_DIR / "retrieval_config_nb03_final_hybrid_static_rag.json",
    "nb03_results": DATA_DIR / "retrieval_results_nb03_final_hybrid_static_rag.csv",
    "nb04b_config": DATA_DIR / "nb04b_context_repaired_experiment_config.json",
    "nb04b_answers": DATA_DIR / "nb04b_context_repaired_openai_luna_answers_comparison.csv",
}
for key, path in FILES.items():
    assert path.is_file(), f"Missing {key}: {path}"

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"], encoding="utf-8-sig")
questions = benchmark["questions"]
Q8 = questions["Q8"]
benchmark_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert len(legal_chunks) == 1383, "Library changed: investigate before running controlled test"
assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["top_k"] == 5
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert set(comparison_df.qid) == set(questions)

FROZEN_Q8_ANSWER = comparison_df.loc[comparison_df.qid=="Q8", "rag_answer"].iloc[0]
print("Benchmark:", benchmark["benchmark_id"])
print("Chunks:", len(legal_chunks), "| Q8:", Q8)
print("Frozen NB04b Q8 answer loaded, characters:", len(FROZEN_Q8_ANSWER))

Benchmark: regulatory_agent_v1_notion_2026-10-08
Chunks: 1383 | Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?
Frozen NB04b Q8 answer loaded, characters: 2752


## 2. **Exact NB03 retriever implementation** (carry forward NB05 v2)

This implementation mirrors original NB03:
- searchable text = title/file/location/article/heading/text
- **BM25 tokenizer** = `[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?`
- normalized embeddings + dot-product, BM25Okapi
- Dense 50 + BM25 50, RRF k=60, source-group cap 2
- rank-ordered Top-5 for frozen test queries.

**No new BM25 or retrieval enhancement is introduced in NB06.**

In [3]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [safe(chunk.get(k)) for k in (
        "regulation_title", "file", "location", "article", "heading", "text"
    )]
    return " | ".join(p for p in parts if p).strip()

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

legal_texts = [searchable_text(ch) for ch in legal_chunks]
assert all(legal_texts)
EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent retriever components ready:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent retriever components ready: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    return (file_name, article if article else location)

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not (1 <= int(top_k) <= 10):
        raise ValueError("top_k must be in 1..10")
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]
    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)
    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append((idx, chunk, fused_scores[idx]))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break
    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank, "chunk_index": int(idx),
            "chunk_id": chunk.get("chunk_id"), "rrf_score": float(rrf_score),
            "file": chunk.get("file"), "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"), "location": chunk.get("location"),
            "heading": chunk.get("heading"), "text": chunk.get("text"),
        })
    return results

## 3. STOP-GATE · rank-ordered NB03 ↔ NB06 parity: **50/50**

Cannot continue to controlled Q8 if any chunk ID/order differs. Save a separate audit file; frozen NB03 outputs are read-only.

In [5]:
PARITY_PATH = DATA_DIR / "build03_nb06_retrieval_parity_audit.csv"
audit_rows, parity_by_qid = [], {}
for qid, question in questions.items():
    baseline = nb03_df[nb03_df.qid == qid].sort_values("rank")
    expected = baseline.chunk_id.astype(str).tolist()
    actual = [r["chunk_id"] for r in hybrid_search(question, top_k=5)]
    assert len(expected) == len(actual) == 5
    parity_by_qid[qid] = expected == actual
    for rank, (a,b) in enumerate(zip(expected,actual), start=1):
        audit_rows.append({"qid":qid,"rank":rank,"nb03_chunk_id":a,"nb06_chunk_id":b,"match":a==b})
    print(f"{qid}: {'PASS' if parity_by_qid[qid] else 'MISMATCH'}")
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(PARITY_PATH, index=False, encoding="utf-8-sig")
matched = int(audit_df.match.sum())
PARITY_OK = matched == 50 and all(parity_by_qid.values())
print(f"\nPARITY: {matched}/50 ordered chunks | Saved: {PARITY_PATH}")
assert PARITY_OK, "STOP: original NB03 retrieval parity failed. No official Q8 run allowed."

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks | Saved: data\build03_nb06_retrieval_parity_audit.csv


## 4. Frozen law registry

No changes to the five input laws, snapshot dates or original corpus. External lookups are restricted to official EU Publications Office Cellar identifiers.


In [6]:
LAW_REGISTRY = {
    "2023/915": {"celex":"32023R0915", "eli":"reg/2023/915", "snapshot_date":"2026-08-19", "snapshot_file":"EU_2023_915_consolidated_2026-08-19.html", "title":"Contaminants maximum levels"},
    "2019/1793": {"celex":"32019R1793", "eli":"reg_impl/2019/1793", "snapshot_date":"2026-06-30", "snapshot_file":"EU_2019_1793_consolidated_2026-06-30.html", "title":"Increased import controls / special conditions"},
    "2017/625": {"celex":"32017R0625", "eli":"reg/2017/625", "snapshot_date":"2025-01-05", "snapshot_file":"EU_2017_625_consolidated_2025-01-05.html", "title":"Official controls"},
    "178/2002": {"celex":"32002R0178", "eli":"reg/2002/178", "snapshot_date":"2026-01-01", "snapshot_file":"EU_2002_178_consolidated_2026-01-01.html", "title":"General Food Law"},
    "2019/1013": {"celex":"32019R1013", "eli":"reg_impl/2019/1013", "snapshot_date":None, "snapshot_file":"EU_2019__1013_original.html", "title":"Prior-notification arrangements"},
}

for key, item in LAW_REGISTRY.items():
    assert (DATA_DIR/item["snapshot_file"]).is_file(), f"Snapshot missing for {key}"
    item["official_basic_url"] = f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A{item["celex"]}'

assert set(LAW_REGISTRY) == {"2023/915","2019/1793","2017/625","178/2002","2019/1013"}
print("Current-law checker scope:", list(LAW_REGISTRY))

Current-law checker scope: ['2023/915', '2019/1793', '2017/625', '178/2002', '2019/1013']


## 5. Official Cellar SPARQL metadata client (v2)

EUR-Lex returned HTTP 202 with an empty HTML body in the previous run. This implementation queries official Cellar SPARQL for consolidated works by CELEX. The original v1 SPARQL draft mistakenly queried `work_id_document`; Cellar legal CELEX work identifiers use `resource_legal_id_celex`. It checks two retrieval paths and validates consolidated CELEX prefixes, dates, result shape, HTTP errors and redirects. **A latest date candidate is not proof of legal applicability or exhaustive coverage.**

**v3 correction:** A live diagnostic using `FILTER(CONTAINS(STR(?celex), "2019R1793"))` returned 19 records, including `02019R1793-20260630`. The prior `bif:contains` index query silently returned zero. The index query now uses the successful portable SPARQL pattern. A zero-row relation lookup is not itself an error, and an index match remains metadata evidence, not article-level legal verification.


In [7]:
# Official Cellar SPARQL: CELEX identifiers are stored under
# cdm:resource_legal_id_celex (NOT cdm:work_id_document).
# No EUR-Lex HTML scraping, no user-supplied URLs, no fabricated latest dates.
from urllib.parse import urlparse

CELLAR_SPARQL = "https://publications.europa.eu/webapi/rdf/sparql"
CELLAR_CELEX_BASE = "https://publications.europa.eu/resource/celex/"
CELLAR_TIMEOUT = (10, 45)
MAX_SPARQL_BYTES = 2_000_000
MAX_VERSIONS = 100


def _base_record(regulation):
    law = LAW_REGISTRY.get(regulation, {})
    return {
        "regulation": regulation, "basic_celex": law.get("celex"),
        "snapshot_date": law.get("snapshot_date"),
        "snapshot_file": law.get("snapshot_file"),
        "official_basic_url": law.get("official_basic_url"),
        "official_metadata_endpoint": CELLAR_SPARQL,
        "checked_utc": datetime.now(timezone.utc).isoformat(),
        "status": "unverified", "source": "EU Publications Office / Cellar SPARQL",
        "latest_date_candidate": None, "latest_celex_candidate": None,
        "observed_consolidations": [], "date_comparison": "not_verified",
        "article_level_changes_verified": False,
        "application_date_verified": False,
        "current_legal_status_verified": False,
        "complete_consolidation_list_verified": False,
    }


def _version_celex_pattern(basic_celex):
    # Sector 3 basic act -> sector 0 consolidated versions.
    return re.compile(r"^0" + re.escape(basic_celex[1:]) + r"-(\d{8})$")


def _sparql_versions_query(celex, mode="relation"):
    assert re.fullmatch(r"3\d{4}R\d{4}", celex), "Only allowlisted regulation CELEX IDs"
    prefix = "0" + celex[1:] + "-"
    header = "PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>\n"
    if mode == "relation":
        # Both documented relation predicates; incoming edges point at base work.
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?original cdm:resource_legal_id_celex "{celex}" .
  {{ ?consolidated cdm:act_consolidated_based_on_resource_legal ?original . }}
  UNION
  {{ ?consolidated cdm:act_consolidated_consolidates_resource_legal ?original . }}
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(STRSTARTS(STR(?consolidated_celex), "{prefix}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    if mode == "celex_index":
        # Validated against live Cellar: the plain SPARQL CONTAINS filter
        # returned 19 CELEX records (including 02019R1793-20260630).
        # The old Virtuoso bif:contains clause silently returned zero rows.
        # Search broadly, then use _parse_versions() to accept ONLY the
        # correct sector-0 consolidated CELEX IDs with valid dates.
        search_fragment = celex[1:]  # e.g. 2019R1793
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(CONTAINS(STR(?consolidated_celex), "{search_fragment}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    raise ValueError("Unknown SPARQL query mode")


def _run_cellar_sparql(query, session=None):
    client = session if session is not None else requests.Session()
    response = client.get(
        CELLAR_SPARQL, params={"query": query},
        headers={"Accept": "application/sparql-results+json",
                 "User-Agent": "ScientificAIBuilder-Research/0.2"},
        timeout=CELLAR_TIMEOUT, allow_redirects=True,
    )
    resolved = getattr(response, "url", CELLAR_SPARQL)
    parsed = urlparse(resolved)
    if parsed.scheme != "https" or parsed.hostname != "publications.europa.eu":
        raise ValueError("untrusted_redirect")
    if response.status_code != 200:
        raise ValueError(f"http_{response.status_code}")
    raw = response.content
    if not raw or len(raw) > MAX_SPARQL_BYTES:
        raise ValueError("empty_or_oversize_response")
    # Virtuoso can return HTTP 200 with plain-text query error.
    content_type = response.headers.get("Content-Type", "").lower()
    if "json" not in content_type:
        raise ValueError("non_json_or_sparql_error")
    data = response.json()
    if not isinstance(data.get("results", {}).get("bindings"), list):
        raise ValueError("invalid_sparql_result_structure")
    return data["results"]["bindings"]


def _parse_versions(bindings, basic_celex, as_of=None):
    as_of = as_of or datetime.now(timezone.utc).date()
    pattern = _version_celex_pattern(basic_celex)
    found = {}
    for row in bindings:
        value = row.get("consolidated_celex", {}).get("value", "")
        match = pattern.fullmatch(value)
        if not match:
            continue
        try:
            dt = datetime.strptime(match.group(1), "%Y%m%d").date()
        except ValueError:
            continue
        # Future-dated consolidations are recorded separately, not current.
        if dt > as_of:
            continue
        found[value] = {
            "celex": value,
            "date": dt.isoformat(),
            "official_celex_url": CELLAR_CELEX_BASE + value,
        }
    return sorted(found.values(), key=lambda v: (v["date"],v["celex"]), reverse=True)


def retrieve_cellar_current_law(regulation, session=None, as_of=None):
    """Official metadata check only; NEVER certify legal force or amended clauses.

    Relational lookup + independently scoped CELEX-index lookup.
    Missing results or failed confirmation are UNVERIFIED, not 'no update'.
    """
    rec = _base_record(regulation)
    if regulation not in LAW_REGISTRY:
        rec["error"] = "not_allowlisted"
        return rec
    basic = LAW_REGISTRY[regulation]["celex"]
    as_of = as_of or datetime.now(timezone.utc).date()
    if isinstance(as_of, str):
        as_of = date.fromisoformat(as_of)
    rec["as_of_date"] = as_of.isoformat()

    try:
        rel = _run_cellar_sparql(_sparql_versions_query(basic,"relation"),session)
        rec["relation_row_count"] = len(rel)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        rel = []
        rec["relation_error"] = f"{type(exc).__name__}: {exc}"

    # Important: index query is independent, not assumed equivalent to links.
    try:
        idx = _run_cellar_sparql(_sparql_versions_query(basic,"celex_index"),session)
        rec["celex_index_row_count"] = len(idx)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        idx = []
        rec["celex_index_error"] = f"{type(exc).__name__}: {exc}"

    observed = _parse_versions(rel+idx,basic,as_of=as_of)
    rec["observed_consolidations"] = observed
    rec["lookup_methods"] = ["relation", "celex_index"]
    if not observed:
        rec["error"] = "no_verified_consolidation_rows"
        return rec

    candidate = observed[0]
    rec["latest_date_candidate"] = candidate["date"]
    rec["latest_celex_candidate"] = candidate["celex"]
    rec["latest_candidate_official_url"] = candidate["official_celex_url"]

    # If either query failed, retrieval may be incomplete.
    if "relation_error" in rec or "celex_index_error" in rec:
        rec["status"] = "partial_metadata"
        rec["date_comparison"] = "latest_observed_only_incomplete_verification"
        return rec
    if len(rel) >= MAX_VERSIONS or len(idx) >= MAX_VERSIONS:
        rec["status"] = "partial_metadata"
        rec["error"] = "result_limit_reached"
        return rec

    rec["status"] = "metadata_found"
    rec["complete_consolidation_list_verified"] = False  # Graph freshness/coverage not provable here.
    snapshot = rec["snapshot_date"]
    if snapshot is None:
        rec["date_comparison"] = "original_snapshot_without_consolidation_date"
    elif candidate["date"] > snapshot:
        rec["date_comparison"] = "newer_consolidation_observed"
    elif candidate["date"] == snapshot:
        rec["date_comparison"] = "matches_latest_observed_consolidation"
    else:
        rec["date_comparison"] = "latest_observed_older_than_snapshot_review"
    return rec

# Alias preserves existing tool wiring below.
retrieve_eurlex_basic_page = retrieve_cellar_current_law


## 6. Offline Function Tests — no network or OpenAI API

Mocked SPARQL responses test normal matching, empty results, HTTP failures, cross-regulation data, future dates, and unknown IDs.


In [8]:
# Offline tests: these DO NOT establish real version metadata.
class _FakeResponse:
    def __init__(self, rows, status=200, url=CELLAR_SPARQL, bad_content_type=False):
        self.status_code = status
        self.url = url
        self.headers = {"Content-Type": "text/html" if bad_content_type else "application/sparql-results+json"}
        self._payload = {"head":{"vars":["consolidated_celex"]},"results":{"bindings":rows}}
        self.content = json.dumps(self._payload).encode("utf-8")
    def json(self): return self._payload

class _FakeSession:
    def __init__(self, responses):
        self.responses = list(responses)
    def get(self,*args,**kwargs):
        return self.responses.pop(0)

def _binding(celex):
    return {"consolidated_celex":{"type":"literal","value":celex}}

valid = _FakeSession([
    _FakeResponse([_binding("02019R1793-20260401")]),
    _FakeResponse([_binding("02019R1793-20260630"),_binding("02019R1793-20260401")]),
])
r = retrieve_cellar_current_law("2019/1793",session=valid,as_of="2026-10-19")
assert r["status"] == "metadata_found" and r["latest_date_candidate"] == "2026-06-30"
assert r["date_comparison"] == "matches_latest_observed_consolidation"
assert r["current_legal_status_verified"] is False

empty = _FakeSession([_FakeResponse([]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=empty,as_of="2026-10-19")
assert r["status"] == "unverified" and r["latest_date_candidate"] is None

blocked = _FakeSession([_FakeResponse([],status=202),_FakeResponse([],status=202)])
r = retrieve_cellar_current_law("2019/1793",session=blocked,as_of="2026-10-19")
assert r["status"] == "unverified"

wrong_law = _FakeSession([_FakeResponse([_binding("02023R0915-20260819")]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=wrong_law,as_of="2026-10-19")
assert r["status"] == "unverified"

future = _FakeSession([_FakeResponse([_binding("02019R1793-20270101")]),_FakeResponse([])])
r = retrieve_cellar_current_law("2019/1793",session=future,as_of="2026-10-19")
assert r["status"] == "unverified"

assert retrieve_cellar_current_law("2099/9999")["error"] == "not_allowlisted"
print("PASS: Cellar SPARQL parsing, exact CELEX identity, dates and failure-closed behavior")

# Regression: use the successful Cellar SPARQL predicate, not Virtuoso full-text bif:contains.
index_query = _sparql_versions_query("32019R1793", "celex_index")
assert 'FILTER(CONTAINS(STR(?consolidated_celex), "2019R1793"))' in index_query
assert "bif:contains" not in index_query
print("PASS: CELEX index query matches successful live diagnostic")


PASS: Cellar SPARQL parsing, exact CELEX identity, dates and failure-closed behavior
PASS: CELEX index query matches successful live diagnostic


## 8. Controlled tools and shared instructions
Both A and B get the **same instructions**. In the matched run, the only controlled manipulation is whether `check_current_law()` appears in the Agent tools list. Separate logs for A/B. The tool itself performs live official requests; its result is **consolidation metadata only**.

In [9]:
# Explicitly create separate tools with experiment-specific logging.
TOOL_LOGS = {"A":{"search":[],"current_law":[]},"B":{"search":[],"current_law":[]}}

@function_tool
def search_regulation_A(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law library for relevant provisions.

    Args:
        query: Focused legal query.
        top_k: Number of result chunks, 1–10.
    """
    results=hybrid_search(query,top_k=max(1,min(int(top_k),10)))
    TOOL_LOGS["A"]["search"].append({"query":query,"top_k":top_k,"results":results})
    return json.dumps([{k:r.get(k) for k in ["rank","chunk_id","file","article","location","heading","text"]} for r in results],ensure_ascii=False)

@function_tool
def search_regulation_B(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law library for relevant provisions.

    Args:
        query: Focused legal query.
        top_k: Number of result chunks, 1–10.
    """
    results=hybrid_search(query,top_k=max(1,min(int(top_k),10)))
    TOOL_LOGS["B"]["search"].append({"query":query,"top_k":top_k,"results":results})
    return json.dumps([{k:r.get(k) for k in ["rank","chunk_id","file","article","location","heading","text"]} for r in results],ensure_ascii=False)

@function_tool
def check_current_law(regulation: str) -> str:
    """Query official EU Cellar metadata for the latest observed consolidation of an allowlisted regulation.

    Args:
        regulation: EU regulation, e.g. 2019/1793 or 2023/915.
    """
    rec=retrieve_cellar_current_law(regulation.strip())
    TOOL_LOGS["B"]["current_law"].append(rec)
    return json.dumps(rec,ensure_ascii=False)

SHARED_CONTEXT=nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT=nb04b_config["system_prompt"]
# Identical for A and B. Mentions metadata possibility without demanding nonexistent tools.
MATCHED_RULES="""Use available tools to ground legal claims in the frozen 5-law corpus.
When available, official Cellar consolidation metadata can supplement local evidence; when unavailable,
state that live currentness was not verified, rather than implying that dates are current.
Identify provisions requiring checks for amendments, entry into force, application date,
transitional rules, annex changes and product-specific classification.
Never infer effective article-level law solely from a matching consolidation date.
Distinguish verified observations from legal conclusions. No invented limits, rates or requirements.
Use 3 headings: 1. Short answer; 2. Legal basis and relevant conditions;
3. Uncertainties / verification needed."""
MATCHED_INSTRUCTIONS="\n\n".join([SHARED_CONTEXT,SHARED_SYSTEM_PROMPT,MATCHED_RULES])
print("Tools and identical A/B instructions ready.")

Tools and identical A/B instructions ready.


## 9. Agent configuration and frozen Q8
No run happens in this cell. Tool aliases have different *function names* for logging; their descriptions and retrieval logic match, but tool names differ (`search_regulation_A` / `_B`) and may independently affect model behavior. This is a practical controlled comparison, **not a perfectly identical tool interface**. Both use exact same benchmark question and output configuration.

In [10]:
assert PARITY_OK and matched==50
Q8=questions["Q8"]
AGENT_KW={"model":MODEL_NAME,"model_settings":ModelSettings(reasoning={"effort":REASONING_EFFORT},max_tokens=int(nb04b_config["max_output_tokens"])),"instructions":MATCHED_INSTRUCTIONS}
agent_A=Agent(name="Build03-A Search Only",tools=[search_regulation_A],**AGENT_KW)
agent_B=Agent(name="Build03-B Search and Current Law",tools=[search_regulation_B,check_current_law],**AGENT_KW)
A_PATH=DATA_DIR/"build03_nb07_q8_A_search_only.json"
B_PATH=DATA_DIR/"build03_nb07_q8_B_matched_tools.json"
NB06_B_PATH=DATA_DIR/"build03_nb06_q8_official.json"
print("Ready | model:",MODEL_NAME,"| effort:",REASONING_EFFORT,"| parity:",matched,"/50")
print("Existing NB06-B:",NB06_B_PATH.exists())
print("Q8:",Q8)

Ready | model: gpt-6-luna | effort: medium | parity: 50 /50
Existing NB06-B: True
Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?


## 10. Run controls (paid API)
Default: **run A only** and reuse existing NB06 B for an exploratory comparison. To isolate the current-law tool more fairly, enable `RUN_MATCHED_B=True` and generate a **new** B using the identical shared instructions. Each run is written once, never overwritten. This cell must run in a Jupyter notebook supporting top-level `await`.

In [11]:
RUN_A=True         # set True for new paid Build03-A call
RUN_MATCHED_B=True  # set True for a new paid matched Build03-B call; requires internet and Cellar access
MAX_TURNS=18

if RUN_A or RUN_MATCHED_B:
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"]=getpass("OPENAI_API_KEY (hidden): ").strip()
    assert os.getenv("OPENAI_API_KEY"),"Key required for paid Agent run"

async def run_once(agent, label, path):
    assert PARITY_OK
    assert not path.exists(),f"Refusing to overwrite: {path}"
    TOOL_LOGS[label]["search"].clear()
    TOOL_LOGS[label]["current_law"].clear()
    result=await Runner.run(agent,Q8,max_turns=MAX_TURNS)
    obj={"experiment":"NB07_Q8_matched_ablation","arm":label,"question_id":"Q8","question":Q8,
         "benchmark_id":benchmark["benchmark_id"],"benchmark_sha256":benchmark_hash,
         "model":MODEL_NAME,"reasoning_effort":REASONING_EFFORT,
         "max_output_tokens":int(nb04b_config["max_output_tokens"]),
         "shared_context":SHARED_CONTEXT,"shared_system_prompt":SHARED_SYSTEM_PROMPT,
         "matched_rules":MATCHED_RULES,"tools_available":(["search_regulation"] if label=="A" else ["search_regulation","check_current_law"]),
         "retrieval_parity":{"matched":matched,"total":50},
         "search_calls":TOOL_LOGS[label]["search"],"current_law_calls":TOOL_LOGS[label]["current_law"],
         "final_output":str(result.final_output),"run_utc":datetime.now(timezone.utc).isoformat(),
         "status":"evaluation_pending"}
    with path.open("x",encoding="utf-8") as f:json.dump(obj,f,ensure_ascii=False,indent=2)
    print("Saved:",path,"| search:",len(obj["search_calls"]),"| law:",len(obj["current_law_calls"]))
    return obj

if RUN_A:
    record_A=await run_once(agent_A,"A",A_PATH)
else:
    print("SKIPPED A; set RUN_A=True for paid run")
if RUN_MATCHED_B:
    live_probe=retrieve_cellar_current_law("2019/1793")
    assert live_probe["status"]=="metadata_found", "STOP: Current-law live probe unverified; avoid paid B official run"
    record_B=await run_once(agent_B,"B",B_PATH)
else:
    print("SKIPPED matched B; set RUN_MATCHED_B=True for strict(er) comparison")

Saved: data\build03_nb07_q8_A_search_only.json | search: 4 | law: 0
Saved: data\build03_nb07_q8_B_matched_tools.json | search: 3 | law: 4


## 11. Frozen Build02 + A + B comparison viewer
Uses frozen Build02 answer and whichever existing files are available. Existing NB06-B is **exploratory only** because its tool-specific instructions differ from NB07. For primary matched comparison, require fresh A and fresh matched B.

In [12]:
def read_json_if(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

A=read_json_if(A_PATH)
B_matched=read_json_if(B_PATH)
B_old=read_json_if(NB06_B_PATH)
print("===== BUILD02 FROZEN Q8 =====\n",FROZEN_Q8_ANSWER)
print("\n===== BUILD03-A (NB07) =====\n",A["final_output"] if A else "NOT RUN")
if B_matched:
    print("\n===== BUILD03-B MATCHED (NB07) =====\n",B_matched["final_output"])
elif B_old:
    print("\n===== BUILD03-B HISTORICAL NB06 (EXPLORATORY ONLY) =====\n",B_old["final_output"])
else:
    print("\n===== BUILD03-B =====\n NOT AVAILABLE")

summary=[{"arm":"Build02 frozen","model":MODEL_NAME,"search_calls":None,"law_calls":0,"instructions_matched_to_A":False},
{"arm":"Build03-A","model":MODEL_NAME,"search_calls":len(A["search_calls"]) if A else None,"law_calls":0,"instructions_matched_to_A":True},
{"arm":"Build03-B matched" if B_matched else "Build03-B historical NB06","model":MODEL_NAME,
 "search_calls":len(B_matched["search_calls"]) if B_matched else (len(B_old.get("local_search_tool_calls",[])) if B_old else None),
 "law_calls":len(B_matched["current_law_calls"]) if B_matched else (len(B_old.get("current_law_tool_calls",[])) if B_old else None),
 "instructions_matched_to_A":bool(B_matched)}]
display(pd.DataFrame(summary))
print("Matched A/B available:",bool(A and B_matched))

===== BUILD02 FROZEN Q8 =====
 ## 1. Short answer

The parts of the assessment that depend on the law in force at the relevant time are the **applicable aflatoxin limits**, the **product’s use or processing classification**, and the **import-control requirements applying to dried figs from Türkiye**. Any conclusions about required certificates or attestations, official checks, or action following a result above a legal limit also need to be checked against the current rules.

The supplied evidence specifically shows that the import-control lists in Regulation (EU) 2019/1793 are reviewed regularly, at intervals not exceeding six months [E1]. It does **not** establish the current limit, whether this product-origin combination is currently listed, or what requirements apply to this shipment.

## 2. Legal basis and relevant conditions

- **Import-control listing and measures:** Check the current Annexes I, II and IIa of Regulation (EU) 2019/1793. The Commission must review those lists regu

,arm,model,search_calls,law_calls,instructions_matched_to_A
0,Build02 frozen,gpt-6-luna,NaN,0,False
1,Build03-A,gpt-6-luna,4.0,0,True
2,Build03-B matched,gpt-6-luna,3.0,4,True


Matched A/B available: True


## 12. Evidence-first grading worksheet
**Do not fabricate grades.** Review model answers against the frozen Verified Reference and actual retrieved chunk IDs. Use the same Correct / Grounded / Complete axes for all three, plus a separate Live Currentness capability dimension. Complete manually; 2019/1013 may have no discoverable consolidated CELEX even where an original act exists.

**Q8 must-haves:** 2023/915 dried figs category/limits/Art 3; 2019/1793 Türkiye dried figs hazard & annex/CN/TARIC; border docs/2017/625/2019/1013; enforcement; effective/application/transition dates; no unsupported claim that snapshot matches prove provisions current.

In [13]:
EVAL_PATH=DATA_DIR/"build03_nb07_q8_evaluation_worksheet.csv"
if EVAL_PATH.exists():
    print("Preserved previous worksheet:",EVAL_PATH)
else:
    pd.DataFrame([{ "question":"Q8","arm":arm,"correct":"REVIEW","grounded":"REVIEW","complete":"REVIEW",
                    "live_metadata_observed":"REVIEW","article_level_currentness":"NOT_VERIFIED",
                    "critical_omissions":"","unsupported_claims":"","reviewer_notes":""}
                  for arm in ["Build02 frozen","Build03-A","Build03-B matched_or_historical"]]).to_csv(EVAL_PATH,index=False,encoding="utf-8-sig")
    print("Created worksheet:",EVAL_PATH)

Created worksheet: data\build03_nb07_q8_evaluation_worksheet.csv


## 13. Experiment interpretation and next stop gate
- **A vs historical NB06-B** is an **exploratory** comparison (instruction and timing differences): cannot causally attribute answer gains solely to the current-law tool.
- **A vs matched B** holds model, efforts, corpus, shared context and instructions constant; tool availability differs. Still non-deterministic; add repeats if you need quantitative inference.
- **Build02 vs A** captures broader change from frozen one-shot RAG to iterative Agent search, **not** agent architecture alone isolated perfectly (different retrieval/query strategy).
- **Do not claim** matching consolidation dates prove actual article-level legal applicability.

### Next: Q1–Q10 full benchmark
**Yes — after reviewing Q8 ablation and fixing known Q7 provision coverage/2019-1013 currentness categorization**, run the remaining benchmark QIDs while preserving frozen Build02 answers and recording separate A/B agent runs. Reuse existing Q7 official run as a *historical* result, but rerun Q7 for strict matched A/B. Use a separate NB08 full-benchmark notebook with cost limits, resumable per-QID files, immutable benchmark and a human rubric. Avoid charging for all 20 runs before reviewing Q8 and Q7.